# Amazon vs Sahel — NDVI Comparison

Side-by-side animation of NDVI (10-daily / dekadal, 300 m)
for two contrasting biomes: the Amazon rainforest and the semi-arid Sahel.


In [ ]:
from functools import partial
from pathlib import Path

from rs_tools.config import BoundingBox
from rs_tools.datasets.loader import load_dataset, load_passes_from_disk
from rs_tools.visualization.animation import save_timeseries_gif_lazy
from rs_tools.visualization.frames import make_dual_panel_composite
from rs_tools.visualization.clms_colormaps import CLMS_NDVI, NDVI_VMIN, NDVI_VMAX

In [ ]:
bbox_amazon = BoundingBox(west=-70, south=-10, east=-50, north=5)
bbox_sahel = BoundingBox(west=-10, south=10, east=15, north=20)
DATA_DIR = "/home/bekaertd/RS_applications/Applications/CGOPS/amazon_vs_sahel"
gif_dir = Path('output/gifs')
gif_dir.mkdir(parents=True, exist_ok=True)

# Which dekads to include: [1], [2], [3], [1,2], etc.  None = all dekads.
DEKADS = None

## Load NDVI dekads for both regions

In [ ]:
amazon_items = load_dataset(
    "CLMS_NDVI_V3", bbox=bbox_amazon,
    start_date="2020-01-01", end_date="2026-03-01",
    limit=150, output_dir=f"{DATA_DIR}/amazon", dekads=DEKADS,
)
print(f"Amazon: {len(amazon_items)} dekads")

In [ ]:
sahel_items = load_dataset(
    "CLMS_NDVI_V3", bbox=bbox_sahel,
    start_date="2020-01-01", end_date="2026-03-01",
    limit=150, output_dir=f"{DATA_DIR}/sahel", dekads=DEKADS,
)
print(f"Sahel: {len(sahel_items)} dekads")

In [ ]:
# Reload as lightweight metadata references (no pixel data in RAM)
amazon_items = load_passes_from_disk(f"{DATA_DIR}/amazon", dekads=DEKADS)
sahel_items = load_passes_from_disk(f"{DATA_DIR}/sahel", dekads=DEKADS)
n = min(len(amazon_items), len(sahel_items))
amazon_items, sahel_items = amazon_items[:n], sahel_items[:n]
print(f"Amazon: {n} dekads  Sahel: {n} dekads")

## Dual-panel GIF (lazy — one frame at a time)

In [ ]:
dual_composite = partial(
    make_dual_panel_composite,
    left_cmap=CLMS_NDVI, right_cmap=CLMS_NDVI,
    left_vmin=NDVI_VMIN, left_vmax=NDVI_VMAX,
    right_vmin=NDVI_VMIN, right_vmax=NDVI_VMAX,
    left_label="Amazon", right_label="Sahel",
)

def _composite(pair):
    left, right = pair
    return dual_composite(left, right)

gif_path = save_timeseries_gif_lazy(
    zip(amazon_items, sahel_items),
    gif_dir / "amazon_vs_sahel_ndvi.gif",
    composite_fn=_composite,
    title="Amazon vs Sahel — NDVI",
    fps=4,
    figsize=(16, 8),
)
print(f"Saved: {gif_path}")